In [ ]:
#Importer pakker
import pandas as pd
import numpy as np

Generelle kommentarer til hvad et datasæt skal opfylde. Forhåbentlig kan denne fil hjælpe med at klargøre datasættet, om ikke andet lede en på rette spor.

Filen indeholder to funktioner:
1. Funktionen "check_dataset(file_name, target_variable)" undersøger ens datasæt og printer resultater.
2. Funktionen "fix_dataset(file_name, target_variable)" implementerer nogle af de ovenstående problemer.
Vær opmærksom på at det her virker for mange datasæt men nok ikke alle.

Generelle guidelines til hvad datasæt skal opfylde:

Krav:
- Skal være en .csv fil
- Første række skal være variablenavne
- Kun numeriske værdier
- Ingen manglende værdier
- Targetværdier skal være kontinuerte (regression)
- Targetværdier skal være 0 eller 1 (classification)

Anbefalinger:
- Sørg for der ikke er alt for få værdier (modellen kommer til at give dårlige resultater) - Balanceret datasæt (jævnt fordelt data/omkring lige mange af alle typer data)
- Vær opmærksom på at outliers kan give dårligere performance






In [ ]:

def check_dataset(file_name, target_variable):
    """
    Comprehensive dataset checker for ML readiness
    
    Parameters:
    -----------
    file_name : str
        Name or path of the CSV file
    target_variable : str
        Name of the target/output column
    """
    print("="*70)
    print(f"DATASET ANALYSIS: {file_name}")
    print("="*70)
    
    try:
        # Load the dataset
        df = pd.read_csv(file_name)
        print(f"✅ File loaded successfully!\n")
        
        # 1. DATASET SIZE
        print("-"*70)
        print("📊 DATASET SIZE")
        print("-"*70)
        print(f"Rows: {len(df)}")
        print(f"Columns: {len(df.columns)}")
        print(f"Column names: {list(df.columns)}\n")
        
        # 2. CHECK IF TARGET VARIABLE EXISTS
        print("-"*70)
        print("🎯 TARGET VARIABLE CHECK")
        print("-"*70)
        if target_variable not in df.columns:
            print(f"❌ ERROR: Target variable '{target_variable}' not found in dataset!")
            print(f"Available columns: {list(df.columns)}")
            return
        else:
            print(f"✅ Target variable '{target_variable}' found!\n")
        
        # 3. MISSING VALUES CHECK
        print("-"*70)
        print("🔍 MISSING VALUES CHECK")
        print("-"*70)
        missing_values = df.isnull().sum()
        total_missing = missing_values.sum()
        
        if total_missing == 0:
            print("✅ No missing values (NaN/NULL) found in any column!")
        else:
            print(f"⚠️ WARNING: Found {total_missing} missing values!")
            print("\nMissing values per column:")
            for col in missing_values[missing_values > 0].index:
                pct = (missing_values[col] / len(df)) * 100
                print(f"  - {col}: {missing_values[col]} ({pct:.2f}%)")
        print()
        
        # 4. DATA TYPES CHECK
        print("-"*70)
        print("🔢 DATA TYPES CHECK")
        print("-"*70)
        print("Column data types:")
        for col in df.columns:
            print(f"  - {col}: {df[col].dtype}")
        
        # Check for non-numeric columns (excluding target for now)
        non_numeric = df.select_dtypes(exclude=[np.number]).columns.tolist()
        if target_variable in non_numeric:
            non_numeric.remove(target_variable)  # We'll check target separately
        
        if non_numeric:
            print(f"\n⚠️ WARNING: Non-numeric columns found: {non_numeric}")
            print("These need to be converted to numeric or removed!")
        else:
            print(f"\n✅ All input columns are numeric!")
        print()
        
        # 5. TARGET VARIABLE ANALYSIS
        print("-"*70)
        print("🎯 TARGET VARIABLE ANALYSIS")
        print("-"*70)
        target_data = df[target_variable]
        unique_values = target_data.nunique()
        unique_list = target_data.unique()
        
        print(f"Target variable: {target_variable}")
        print(f"Data type: {target_data.dtype}")
        print(f"Number of unique values: {unique_values}")
        
        # Determine if binary or continuous
        if unique_values == 2:
            print(f"\n✅ TARGET IS BINARY (Classification problem)")
            print(f"Unique values: {sorted(unique_list)}")
            
            # Check class balance
            value_counts = target_data.value_counts()
            print(f"\nClass distribution:")
            for val, count in value_counts.items():
                pct = (count / len(df)) * 100
                print(f"  - {val}: {count} samples ({pct:.2f}%)")
            
            # Calculate imbalance ratio
            majority_class = value_counts.max()
            minority_class = value_counts.min()
            imbalance_ratio = majority_class / minority_class
            
            print(f"\nImbalance ratio: {imbalance_ratio:.2f}:1")
            if imbalance_ratio < 1.5:
                print("✅ Classes are well balanced!")
            elif imbalance_ratio < 3:
                print("⚠️ Mild class imbalance - should be okay")
            else:
                print("⚠️ WARNING: Significant class imbalance detected!")
                
        elif unique_values > 10 and pd.api.types.is_numeric_dtype(target_data):
            print(f"\n✅ TARGET IS CONTINUOUS (Regression problem)")
            print(f"\nStatistics:")
            print(f"  - Min: {target_data.min():.2f}")
            print(f"  - Max: {target_data.max():.2f}")
            print(f"  - Mean: {target_data.mean():.2f}")
            print(f"  - Median: {target_data.median():.2f}")
            print(f"  - Std: {target_data.std():.2f}")
            
        else:
            print(f"\n⚠️ WARNING: Target has {unique_values} unique values")
            if unique_values <= 10:
                print("This might be multi-class classification (not supported by your code)")
                print(f"Unique values: {sorted(unique_list)}")
            else:
                print("Unable to determine if binary or continuous")
        print()
        
        # 6. OUTLIERS CHECK (for numeric columns)
        print("-"*70)
        print("📈 OUTLIERS CHECK (using IQR method)")
        print("-"*70)
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        outliers_found = False
        
        for col in numeric_cols:
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - 3 * IQR
            upper_bound = Q3 + 3 * IQR
            
            outliers = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()
            if outliers > 0:
                outliers_found = True
                pct = (outliers / len(df)) * 100
                print(f"  - {col}: {outliers} outliers ({pct:.2f}%)")
        
        if not outliers_found:
            print("✅ No extreme outliers detected!")
        print()
        
        # 7. SUMMARY
        print("="*70)
        print("📋 SUMMARY")
        print("="*70)
        
        issues = []
        if total_missing > 0:
            issues.append(f"❌ {total_missing} missing values need handling")
        if non_numeric:
            issues.append(f"❌ {len(non_numeric)} non-numeric columns need conversion")
        if len(df) < 50:
            issues.append(f"⚠️ Small dataset (only {len(df)} rows)")
        
        if not issues:
            print("✅ Dataset looks ready for ML!")
            print("All checks passed - you can proceed with modeling.")
        else:
            print("⚠️ Issues found that need to be addressed:")
            for issue in issues:
                print(f"  {issue}")
        
        print("="*70)
        
    except FileNotFoundError:
        print(f"❌ ERROR: File '{file_name}' not found!")
        print("Please check the file name and path.")
    except Exception as e:
        print(f"❌ ERROR: {str(e)}")



In [ ]:
# Replace with your actual file name and target variable
file_name = "diabetes_data.csv"  # Change this
target_variable = "Diabetes"  # Change this

check_dataset(file_name, target_variable) #Den magiske linje

# Hvis fejlmeddelelserne nævner bestemte rækker i dataen,
# gå gerne ind og kig for at se hvad der kunne være galt.

In [ ]:
def fix_dataset(file_name, target_variable):
    """
    Fix common dataset issues and save cleaned version
    
    Parameters:
    -----------
    file_name : str
        Name or path of the CSV file
    target_variable : str
        Name of the target/output column
    
    Returns:
    --------
    str : Path to the fixed CSV file
    """
    print("="*70)
    print(f"FIXING DATASET: {file_name}")
    print("="*70)
    
    try:
        # Load the dataset
        df = pd.read_csv(file_name)
        original_rows = len(df)
        print(f"✅ Original dataset loaded: {original_rows} rows, {len(df.columns)} columns\n")
        
        # 1. Remove rows with missing values
        print("-"*70)
        print("🧹 REMOVING MISSING VALUES")
        print("-"*70)
        missing_before = df.isnull().sum().sum()
        if missing_before > 0:
            print(f"Found {missing_before} missing values (NaN/NULL)")
            df_cleaned = df.dropna()
            rows_removed = original_rows - len(df_cleaned)
            print(f"✅ Removed {rows_removed} rows with missing values")
            print(f"Remaining rows: {len(df_cleaned)}")
        else:
            df_cleaned = df.copy()
            print("✅ No missing values found - no rows removed")
        print()
        
        # 2. Remove non-numeric columns (except target)
        print("-"*70)
        print("🔢 CHECKING DATA TYPES")
        print("-"*70)
        non_numeric_cols = df_cleaned.select_dtypes(exclude=[np.number]).columns.tolist()
        
        # Keep target even if non-numeric (we'll handle it separately)
        cols_to_remove = [col for col in non_numeric_cols if col != target_variable]
        
        if cols_to_remove:
            print(f"⚠️ Removing {len(cols_to_remove)} non-numeric columns:")
            for col in cols_to_remove:
                print(f"  - {col} ({df_cleaned[col].dtype})")
            df_cleaned = df_cleaned.drop(columns=cols_to_remove)
            print(f"✅ Removed non-numeric columns")
        else:
            print("✅ All columns are numeric - nothing to remove")
        print()
        
        # 3. Check target variable
        print("-"*70)
        print("🎯 TARGET VARIABLE CHECK")
        print("-"*70)
        if target_variable not in df_cleaned.columns:
            print(f"❌ ERROR: Target variable '{target_variable}' was removed or doesn't exist!")
            return None
        
        # Convert target to numeric if possible
        if not pd.api.types.is_numeric_dtype(df_cleaned[target_variable]):
            print(f"⚠️ Target variable '{target_variable}' is non-numeric")
            try:
                df_cleaned[target_variable] = pd.to_numeric(df_cleaned[target_variable])
                print(f"✅ Successfully converted target to numeric")
            except:
                print(f"❌ ERROR: Cannot convert target to numeric - please check your data")
                return None
        else:
            print(f"✅ Target variable is numeric")
        print()
        
        # 4. Remove duplicate rows
        print("-"*70)
        print("🔍 CHECKING FOR DUPLICATES")
        print("-"*70)
        duplicates_before = df_cleaned.duplicated().sum()
        if duplicates_before > 0:
            df_cleaned = df_cleaned.drop_duplicates()
            print(f"✅ Removed {duplicates_before} duplicate rows")
        else:
            print("✅ No duplicate rows found")
        print()
        
        # 5. Final statistics
        print("-"*70)
        print("📊 FINAL DATASET STATISTICS")
        print("-"*70)
        print(f"Original rows: {original_rows}")
        print(f"Final rows: {len(df_cleaned)}")
        print(f"Rows removed: {original_rows - len(df_cleaned)} ({((original_rows - len(df_cleaned))/original_rows)*100:.1f}%)")
        print(f"Final columns: {len(df_cleaned.columns)}")
        print(f"Column names: {list(df_cleaned.columns)}")
        print()
        
        # 6. Save cleaned dataset
        print("-"*70)
        print("💾 SAVING CLEANED DATASET")
        print("-"*70)
        
        # Create output filename
        import os
        file_dir = os.path.dirname(file_name)
        file_base = os.path.basename(file_name)
        file_name_without_ext = os.path.splitext(file_base)[0]
        output_filename = os.path.join(file_dir, f"{file_name_without_ext}_fixed.csv")
        
        # Save to CSV
        df_cleaned.to_csv(output_filename, index=False)
        print(f"✅ Cleaned dataset saved as:")
        print(f"   {output_filename}")
        print()
        
        # 7. Summary
        print("="*70)
        print("📋 SUMMARY")
        print("="*70)
        
        if len(df_cleaned) < 20:
            print("⚠️ WARNING: Very small dataset after cleaning!")
            print(f"   Only {len(df_cleaned)} rows remaining - may not be enough for ML")
        elif len(df_cleaned) < 100:
            print("⚠️ WARNING: Small dataset after cleaning")
            print(f"   {len(df_cleaned)} rows - results may vary")
        else:
            print(f"✅ Dataset cleaned successfully!")
            print(f"   {len(df_cleaned)} rows ready for machine learning")
        
        print("="*70)
        
        return output_filename
        
    except FileNotFoundError:
        print(f"❌ ERROR: File '{file_name}' not found!")
        print("Please check the file name and path.")
        return None
    except Exception as e:
        print(f"❌ ERROR: {str(e)}")
        import traceback
        traceback.print_exc()
        return None



In [ ]:

# Usage:
# First run check_dataset to see what issues exist
# Then run fix_dataset to create a cleaned version

file_name = "Diabetes_data.csv"  # Change this
target_variable = "Diabetes"

# Fix the dataset
fixed_file = fix_dataset(file_name, target_variable) # Den magiske linje

# If successful, check the fixed dataset
if fixed_file:
    print("\n" + "="*70)
    print("VERIFYING FIXED DATASET")
    print("="*70 + "\n")
    check_dataset(fixed_file, target_variable)